In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import joblib
import yfinance as yf
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

In [3]:
TICKERS = {
    "Naspers":        "NPN.JO",
    "MTN Group":      "MTN.JO",
    "Standard Bank":  "SBK.JO",
    "Anglo American": "AGL.JO",
    "Shoprite":       "SHP.JO"
}

#Load saved models
garch_models = {}

for stock in TICKERS.keys():
    filename = stock.lower().replace(" ", "_")
    garch_models[stock] = joblib.load(f"../models/garch_{filename}.pkl")
    print(f"✅ Loaded garch_{filename}.pkl")

print(f"\n {len(garch_models)} models loaded successfully")

✅ Loaded garch_naspers.pkl
✅ Loaded garch_mtn_group.pkl
✅ Loaded garch_standard_bank.pkl
✅ Loaded garch_anglo_american.pkl
✅ Loaded garch_shoprite.pkl

 5 models loaded successfully


In [ ]:
# Downloaded 2025 test data for backtesting the GARCH models. We use the same tickers and a full year to ensure we have enough data points for evaluation.

raw_test = yf.download(
    list(TICKERS.values()),
    start="2025-01-01",
    end="2025-12-31",
    auto_adjust=True
)

test_prices = raw_test["Close"]
test_prices.columns = list(TICKERS.keys())
test_prices.dropna(how="all", inplace=True)

# Calculate log returns
test_returns = np.log(test_prices / test_prices.shift(1)).dropna()
test_pct     = test_returns * 100   # scale to percentage

print(f"\n Fresh test data downloaded")
print(f"   Period : {test_pct.index[0].date()} → {test_pct.index[-1].date()}")
print(f"   Shape  : {test_pct.shape}")
# first 5 rows of test returns in percentage
print(test_pct.head().round(4))

[*********************100%***********************]  5 of 5 completed


 Fresh test data downloaded
   Period : 2025-01-03 → 2025-12-30
   Shape  : (248, 5)
            Naspers  MTN Group  Standard Bank  Anglo American  Shoprite
Date                                                                   
2025-01-03  -0.5296     0.8066         0.6999          1.0015    0.8151
2025-01-06   1.2009     0.6725         0.4329          0.4084   -0.0671
2025-01-07   0.0487    -0.7368       -10.6428         -0.0931    1.6810
2025-01-08  -0.6872    -4.8421        -2.4268         -2.5011   -1.8523
2025-01-09   2.8816     1.7508         1.1822          0.2860   -1.0581


In [7]:
# We use the saved model parameters to forecast
# volatility for each day in the test period

VaR_95 = {}
VaR_99 = {}
forecast_vol = {}

for stock, model in garch_models.items():
    print(f"\nForecasting volatility → {stock}")

    var95_list = []
    var99_list = []
    vol_list   = []

    # Extract model parameters (omega, alpha, beta)
    params = model.params
    omega  = params["omega"]
    alpha  = params["alpha[1]"]
    beta   = params["beta[1]"]
    mu     = params["mu"]

    # Last known variance from training period
    last_variance = model.conditional_volatility.iloc[-1] ** 2

    # Use GARCH recursion formula to forecast each test day
    # σ²_t = ω + α * ε²_(t-1) + β * σ²_(t-1)
    current_variance = last_variance

    for i in range(len(test_pct)):
        # Forecast next day sigma using GARCH recursion
        sigma = np.sqrt(current_variance)
        vol_list.append(sigma)

        # VaR at 95% and 99% confidence
        var95_list.append(mu - stats.norm.ppf(0.95) * sigma)
        var99_list.append(mu - stats.norm.ppf(0.99) * sigma)

        # Update variance using today's actual return (if available)
        actual_return      = test_pct[stock].iloc[i]
        epsilon_sq         = (actual_return - mu) ** 2
        current_variance   = omega + alpha * epsilon_sq + beta * current_variance

    VaR_95[stock]       = pd.Series(var95_list, index=test_pct.index)
    VaR_99[stock]       = pd.Series(var99_list, index=test_pct.index)
    forecast_vol[stock] = pd.Series(vol_list,   index=test_pct.index)

    print(f"   Avg forecast vol : {np.mean(vol_list):.4f}%")
    print(f"   Avg VaR 95%      : {np.mean(var95_list):.4f}%")
    print(f"   Avg VaR 99%      : {np.mean(var99_list):.4f}%")


Forecasting volatility → Naspers
   Avg forecast vol : 2.2069%
   Avg VaR 95%      : -3.5506%
   Avg VaR 99%      : -5.0546%

Forecasting volatility → MTN Group
   Avg forecast vol : 2.4956%
   Avg VaR 95%      : -4.1383%
   Avg VaR 99%      : -5.8391%

Forecasting volatility → Standard Bank
   Avg forecast vol : 2.1777%
   Avg VaR 95%      : -3.5265%
   Avg VaR 99%      : -5.0107%

Forecasting volatility → Anglo American
   Avg forecast vol : 23.4979%
   Avg VaR 95%      : -38.6135%
   Avg VaR 99%      : -54.6272%

Forecasting volatility → Shoprite
   Avg forecast vol : 1.4555%
   Avg VaR 95%      : -2.3895%
   Avg VaR 99%      : -3.3814%


In [11]:
backtest_rows = []

for stock in TICKERS.keys():
    actual = test_pct[stock]
    v95    = VaR_95[stock]
    v99    = VaR_99[stock]
    n      = len(actual)

    breaches_95    = (actual < v95).sum()
    breaches_99    = (actual < v99).sum()
    breach_rate_95 = breaches_95 / n
    breach_rate_99 = breaches_99 / n

    backtest_rows.append({
        "Stock":                 stock,
        "Test Days":             n,
        "Expected 95%":          round(n * 0.05),
        "Actual 95% breaches":   breaches_95,
        "Breach Rate 95%":       round(breach_rate_95, 4),
        "Expected 99%":          round(n * 0.01),
        "Actual 99% breaches":   breaches_99,
        "Breach Rate 99%":       round(breach_rate_99, 4),
    })

summary_df = pd.DataFrame(backtest_rows).set_index("Stock")

print("\n Backtest Results")
print("="*80)
print(summary_df.to_string())
print("\n Good model → actual breaches ≈ expected breaches")
print(" Too many breaches  → model underestimates risk (dangerous)")
print(" Too few breaches   → model is too conservative")


 Backtest Results
                Test Days  Expected 95%  Actual 95% breaches  Breach Rate 95%  Expected 99%  Actual 99% breaches  Breach Rate 99%
Stock                                                                                                                            
Naspers               248            12                    8           0.0323             2                    2           0.0081
MTN Group             248            12                    5           0.0202             2                    3           0.0121
Standard Bank         248            12                    7           0.0282             2                    2           0.0081
Anglo American        248            12                    7           0.0282             2                    4           0.0161
Shoprite              248            12                    9           0.0363             2                    4           0.0161

 Good model → actual breaches ≈ expected breaches
 Too many breaches  

In [14]:
def kupiec_test(n, breaches, confidence):
    """
    Kupiec Proportion of Failures test.
    H0: breach rate = expected rate
    p > 0.05 → model passes
    """
    expected_rate = 1 - confidence
    actual_rate   = breaches / n

    if breaches == 0:
        return np.nan, "⚠️  No breaches"

    if breaches == n:
        return np.nan, "⚠️  All days breached"

    lr = -2 * (
        np.log((1 - expected_rate) ** (n - breaches) * expected_rate ** breaches) -
        np.log((1 - actual_rate)  ** (n - breaches) * actual_rate  ** breaches)
    )

    p_value = 1 - stats.chi2.cdf(lr, df=1)
    result  = "✅ PASS" if p_value > 0.05 else "❌ FAIL"
    return round(p_value, 4), result


print("\n Kupiec POF Test Results")
print("="*60)
print(f"{'Stock':<18} {'VaR 95%':<25} {'VaR 99%'}")
print("-"*60)

for stock in TICKERS.keys():
    actual = test_pct[stock]
    n      = len(actual)
    b95    = (actual < VaR_95[stock]).sum()
    b99    = (actual < VaR_99[stock]).sum()

    p95, r95 = kupiec_test(n, b95, 0.95)
    p99, r99 = kupiec_test(n, b99, 0.99)

    print(f"{stock:<18} p={p95}  {r95:<15} p={p99}  {r99}")

print("\n p > 0.05 = breach rate is statistically acceptable")


 Kupiec POF Test Results
Stock              VaR 95%                   VaR 99%
------------------------------------------------------------
Naspers            p=0.1715  ✅ PASS          p=0.7512  ✅ PASS
MTN Group          p=0.0147  ❌ FAIL          p=0.748  ✅ PASS
Standard Bank      p=0.0876  ✅ PASS          p=0.7512  ✅ PASS
Anglo American     p=0.0876  ✅ PASS          p=0.373  ✅ PASS
Shoprite           p=0.2986  ✅ PASS          p=0.373  ✅ PASS

 p > 0.05 = breach rate is statistically acceptable


In [17]:
COLORS = ["#00ff88", "#ff6b6b", "#4ecdc4", "#ffe66d", "#a29bfe"]

In [26]:
# Save backtest summary
summary_df.to_csv("../data/backtest_results.csv")
print("Saved backtest_results.csv")

# Save VaR forecasts
var95_df = pd.DataFrame(VaR_95)
var99_df = pd.DataFrame(VaR_99)
vol_df   = pd.DataFrame(forecast_vol)

var95_df.to_csv("../data/var_95_forecasts.csv")
var99_df.to_csv("../data/var_99_forecasts.csv")
vol_df.to_csv("../data/volatility_forecasts.csv")

print("Saved VaR forecasts")
print("Saved volatility forecasts")
print("\nStep 4 complete — models tested on unseen 2024 data")

Saved backtest_results.csv
Saved VaR forecasts
Saved volatility forecasts

Step 4 complete — models tested on unseen 2024 data
